
# PRÁCTICA 1: FUNDAMENTOS DEL PLN



## CONFIGURACIÓN INICIAL
NLTK: https://www.nltk.org/
Spacy: https://spacy.io/

In [2]:
# Ejecuta esta celda para instalar y cargar las dependencias necesarias.
!pip install pandas nltk spacy
!python -m spacy download es_core_news_md

import pandas as pd
import nltk
import spacy
import re
import heapq
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Descargas necesarias de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Cargar el motor de spaCy (tamaño medio para mayor precisión)
nlp = spacy.load("es_core_news_md")

print("¡Entorno configurado correctamente!")

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
     --------------------------------------- 42.3/42.3 MB 17.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')



[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jperaltaza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jperaltaza\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jperaltaza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


¡Entorno configurado correctamente!


## FASE 1: Limpieza de Datos (Data Cleaning)

In [3]:
texto_raw = """
<p>¡Increíble servicio de @TechStore! 🤩 Compré el nuevo portátil por 1500€ en https://ejemplo.com/portatil.
El envío llegó a MÁLAGA en 24h. Totalmente recomendado... ¿volveré a comprar? ¡Seguro!</p>
"""

def limpiar_texto_avanzado(texto):
    reemplazo = ""
    # PISTA: Usa re.sub(patron, reemplazo, texto) para ir "pisando" la variable texto en cada paso.

    # 1. Eliminar URLs (Pista: busca 'http\S+' o 'www.\S+')
    patron = r"http\S+"
    texto = re.sub(patron, reemplazo, texto)
    # 2. Eliminar etiquetas HTML (Pista: '<[^>]+>')
    patron = r"<[^>]+>"
    texto = re.sub(patron, reemplazo, texto)
    # 3. Eliminar menciones (@usuario)
    patron = r"@\S+"
    texto = re.sub(patron, reemplazo, texto)
    # 4. Eliminar todo lo que no sea letra, número o espacio (Pista: '[^\w\s]')
    patron = r"[^\w\s]"
    texto = re.sub(patron, reemplazo, texto)
    # 5. Convertir a minúsculas y quitar espacios extra (Pista: usa .lower(), .split() y " ".join())
    texto = texto.lower().split()
    texto = " ".join(texto)
    return texto

texto_limpio = limpiar_texto_avanzado(texto_raw)
print("--- FASE 1 ---")
print("Texto Limpio:", texto_limpio, "\n")


--- FASE 1 ---
Texto Limpio: increíble servicio de compré el nuevo portátil por 1500 en el envío llegó a málaga en 24h totalmente recomendado volveré a comprar seguro 



## FASE 2 Tokenización y Stop Words

In [4]:
# te damos los tokens y la lista original de stop words
tokens = word_tokenize(texto_limpio, language='spanish') if texto_limpio != texto_raw else []
stop_words_es = set(stopwords.words('spanish'))
palabras_a_mantener = {'no', 'ni', 'tampoco', 'pero'}

# --- TU CÓDIGO AQUÍ ---
# 1. Elimina de 'stop_words_es' las palabras que están en 'palabras_a_mantener'.
# PISTA: Al ser conjuntos (sets), puedes restarlos directamente con el operador '-'.
stop_words = stop_words_es - palabras_a_mantener

# 2. Crea la lista 'tokens_filtrados' iterando sobre 'tokens' y guardando solo
#    los que NO estén en tu nuevo conjunto de stop_words.
tokens_filtrados = []
for token in tokens:
    if stop_words.__contains__(token) == False:
        tokens_filtrados.append(token)


print("--- FASE 2 ---")
print("Tokens tras filtrado:", tokens_filtrados, "\n")

--- FASE 2 ---
Tokens tras filtrado: ['increíble', 'servicio', 'compré', 'nuevo', 'portátil', '1500', 'envío', 'llegó', 'málaga', '24h', 'totalmente', 'recomendado', 'volveré', 'comprar', 'seguro'] 



## FASE 3 - NORMALIZACION (Lematización contextual)

In [5]:
texto_para_lematizar = "Los gatos están corriendo rápido porque comieron mucho"
doc = nlp(texto_para_lematizar)

print("--- FASE 3 ---")
print(f"{'Token':<12} | {'Lema':<12} | {'Categoría (POS)'}")
print("-" * 45)

# --- TU CÓDIGO AQUÍ ---
# Recorre el objeto 'doc' (que contiene los tokens analizados por spaCy).
# Imprime el texto del token (.text), su lema (.lemma_) y su categoría gramatical (.pos_).
# PISTA: Utiliza la misma estructura de formato f-string del print superior para que quede en tabla.
for token in doc:
    print(f"{token.text:<12} | {token.lemma:<12} | {token.pos_}")

--- FASE 3 ---
Token        | Lema         | Categoría (POS)
---------------------------------------------
Los          | 11488171005156075516 | DET
gatos        | 9565357104409163886 | NOUN
están        | 6576417655588717493 | AUX
corriendo    | 10468945644875678183 | VERB
rápido       | 9694796508432521733 | ADV
porque       | 5757082442330786044 | SCONJ
comieron     | 5453621257116136642 | VERB
mucho        | 3038928633385455635 | ADV


## FASE 4 - Extracción de Información Crítica (NER)

In [6]:
# ==============================================================================
# FASE 4: Extracción de Información Crítica (NER)
# ==============================================================================
noticia = "El Banco Santander anunció ayer en Madrid la compra de una startup alemana por 45 millones de euros, superando a BBVA."
doc_ner = nlp(noticia)

print("--- FASE 4 ---")
# --- TU CÓDIGO AQUÍ ---
# Itera sobre las entidades encontradas en el documento (doc_ner.ents).
# Imprime el texto de la entidad (.text) y el tipo de entidad (.label_).
for token in doc_ner:
    print(f"{token.text:<12} | {token.ent_type_}")

--- FASE 4 ---
El           | 
Banco        | ORG
Santander    | ORG
anunció      | 
ayer         | 
en           | 
Madrid       | LOC
la           | 
compra       | 
de           | 
una          | 
startup      | 
alemana      | 
por          | 
45           | 
millones     | 
de           | 
euros        | 
,            | 
superando    | 
a            | 
BBVA         | ORG
.            | 


# ♦ RETOS ♦

### Reto 1: Construcción de un Pipeline de Producción en Pandas
Objetivo: crear una función pipeline_pln que limpie y procese texto utilizando spaCy y NLTK, filtrando stop words, puntuación y números. Cuando la tengais, debeis  aplicar esta función a una columna de un DataFrame de pandas para generar una nueva columna con el texto procesado.

In [7]:
datos = pd.DataFrame({
    'id_ticket': [1, 2, 3],
    'comentario': [
        "El router WiFi ZTE-450 no funciona desde ayer, luz roja parpadeando.",
        "Deseo cancelar mi suscripción premium de 15€ a Spotify inmediatamente.",
        "¡Excelente cobertura en las afueras de Barcelona! Muy contento 5/5."
    ]
})

def pipeline_pln(texto):
    # --- TU CÓDIGO AQUÍ ---
    # 1. Llama a tu función limpiar_texto_avanzado()
    texto = limpiar_texto_avanzado(texto)
    # 2. Pasa el resultado por el modelo nlp()
    doc = nlp(texto)
    # 3. Itera los tokens y guarda en una lista sus lemas SOLO SI cumplen 3 condiciones:
    #    - No son stop words
    #    - No son puntuación
    #    - No son números
    tokens_filtrados = []
    for token in doc:
        if not token.is_stop and not token.is_punct and not token.like_num:
            tokens_filtrados.append(token.lemma_)
    # 4. Devuelve la lista de lemas unida en un solo string (usa " ".join())
    return " ".join(tokens_filtrados)

# Aplica la función a la columna 'comentario' usando .apply() y guarda el resultado
# en una nueva columna llamada 'comentario_procesado'
datos['comentario_procesado'] = datos['comentario'].apply(pipeline_pln)

print("--- RETO 1 ---")
print(datos[['comentario', 'comentario_procesado']])
print("\n")


--- RETO 1 ---
                                          comentario  \
0  El router WiFi ZTE-450 no funciona desde ayer,...   
1  Deseo cancelar mi suscripción premium de 15€ a...   
2  ¡Excelente cobertura en las afueras de Barcelo...   

                                comentario_procesado  
0  router wifi zte450 funcionar ayer luz rojo par...  
1  desear cancelar suscripción premium spotify in...  
2      excelente cobertura afuera barcelona contento  




### RETO 2: Inteligencia Competitiva (NLP Analítico)
OBJETIVO: Crear un diccionario donde las claves sean las marcas (ORG) y los valores sean una lista de los adjetivos (ADJ) asociados a ellas en su oración.


In [8]:
texto_mercado = """
Vodafone tiene una red rápida y muy estable, pero su atención al cliente es pésima.
Por otro lado, Movistar ofrece servicios caros, aunque su fibra es excelente y fiable.
Orange lanzó una tarifa barata y atractiva para los estudiantes.
"""
doc_mercado = nlp(texto_mercado)
analisis_marcas = {}

# --- TU CÓDIGO AQUÍ ---
# PISTA 1: Itera oración por oración usando doc_mercado.sents
for oracion in doc_mercado.sents:
# PISTA 2: Dentro de cada oración, busca las entidades (ent.label_ == "ORG")
    marcas = []
    for ent in oracion.ents:
        if ent.label_ == "ORG":
            marcas.append(ent.text)
# PISTA 3: Dentro de la misma oración, busca los tokens que sean adjetivos (token.pos_ == "ADJ")
    adjetivos = []
    for token in oracion:
        if token.pos_ == "ADJ":
            adjetivos.append(token.lemma_.lower())
# PISTA 4: Añade los adjetivos encontrados a la lista correspondiente de la marca en el diccionario.
    for marca in marcas:
        if marca not in analisis_marcas:
            analisis_marcas[marca] = []
        analisis_marcas[marca].extend(adjetivos)

print("--- RETO 2 ---")
print(analisis_marcas)
print("\n")

--- RETO 2 ---
{'Vodafone': ['rápido', 'estable', 'pésimo'], 'Movistar': ['otro', 'caro', 'excelente', 'fiable'], 'Orange': ['barato', 'atractivo']}




### RETO 3: Análisis de Sentimiento basado en Léxico

In [13]:
def clasificador_sentimiento_basico(texto_lematizado):
    palabras_pos = ['excelente', 'rápido', 'bueno', 'fiable', 'barato', 'atractivo', 'contento', 'estable']
    palabras_neg = ['pésimo', 'caro', 'malo', 'lento', 'error', 'fallo']

    # --- TU CÓDIGO AQUÍ ---
    # 1. Inicia una variable 'score' en 0.
    score = 0

    # 2. Divide 'texto_lematizado' en palabras (.split()) y recórrelas.
    for palabra in texto_lematizado.split():
        # 3. Si la palabra está en palabras_pos, suma 1 al score. Si está en palabras_neg, resta 1.
        if palabra in palabras_pos:
            score += 1
        elif palabra in palabras_neg:
            score -= 1

    # 4. Devuelve "Positivo" si score > 0, "Negativo" si score < 0, o "Neutro" si es 0.
    if score > 0:
        return "Positivo"
    elif score < 0:
        return "Negativo"
    return "Neutro"

texto_prueba = "El servicio es muy rápido y bueno, pero el precio es bastante caro."
# PISTA: Pasa el texto_prueba por tu pipeline_pln antes de clasificarlo.
texto_prueba_procesado = pipeline_pln(texto_prueba)

print("--- RETO 3 ---")
print(f"Texto original: {texto_prueba}")
print(f"Texto procesado: {texto_prueba_procesado}")
print(f"Sentimiento Calculado: {clasificador_sentimiento_basico(texto_prueba_procesado)}")
print("\n")

--- RETO 3 ---
Texto original: El servicio es muy rápido y bueno, pero el precio es bastante caro.
Texto procesado: servicio rápido precio caro
Sentimiento Calculado: Neutro




### Reto 4: Resumen Automático por Frecuencia (NLG Básico)

In [15]:
texto_noticia = """
La inteligencia artificial generativa está transformando la industria a pasos agigantados.
ChatGPT entiende lo que escribes gracias al procesamiento de lenguaje natural.
El PLN es la rama de la IA enfocada en la interacción con el lenguaje humano.
Permite a los sistemas analizar texto de forma coherente y contextual.
Actualmente, el 80% de los datos mundiales son texto no estructurado.
"""

def generar_resumen_frecuencia(texto_largo, n_oraciones=2):
    doc_resumen = nlp(texto_largo)

    # --- TU CÓDIGO AQUÍ ---
    # PASO 1: Crea un diccionario de frecuencias de palabras (ignorando stop words y puntuación).
    frecuencias = {}
    for token in doc_resumen:
        if not token.is_stop and not token.is_punct:
            palabra = token.lemma_.lower()
            if palabra in frecuencias:
                frecuencias[palabra] += 1
            else:
                frecuencias[palabra] = 1
    # PASO 2: Encuentra la frecuencia máxima y divide todos los valores entre ella para normalizar de 0 a 1.
    max_frecuencia = max(frecuencias.values())
    for palabra in frecuencias:
        frecuencias[palabra] = frecuencias[palabra] / max_frecuencia
    # PASO 3: Recorre las oraciones (doc_resumen.sents). Puntúa cada oración sumando el valor normalizado de las palabras que contiene.
    oraciones_puntuadas = {}
    for oracion in doc_resumen.sents:
        puntuacion = 0
        for token in oracion:
            palabra = token.lemma_.lower()
            if palabra in frecuencias:
                puntuacion += frecuencias[palabra]
        oraciones_puntuadas[oracion] = puntuacion
    # PASO 4: Usa heapq.nlargest para obtener las oraciones con mayor puntuación.
    # Puedes consultar la documentación de heapq en Python.
    oraciones_resumen = heapq.nlargest(n_oraciones, oraciones_puntuadas, key=oraciones_puntuadas.get)

    return oraciones_puntuadas

print("--- RETO 4 ---")
print(generar_resumen_frecuencia(texto_noticia))
print("\n")

--- RETO 4 ---
{
: 1.0, La inteligencia artificial generativa está transformando la industria a pasos agigantados.
: 2.1666666666666665, ChatGPT entiende lo que escribes gracias al procesamiento de lenguaje natural.
: 2.333333333333333, El PLN es la rama de la IA enfocada en la interacción con el lenguaje humano.
: 2.333333333333333, Permite a los sistemas analizar texto de forma coherente y contextual.
: 2.333333333333333, Actualmente, el 80% de los datos mundiales son texto no estructurado.
: 2.166666666666667}




### RETO 5 Anonimización de Datos Sensibles
OBJETIVO: Reemplazar emails por [EMAIL], teléfonos por [TELÉFONO] y nombres por [PERSONA].

In [18]:
texto_confidencial = """
Incidencia #4092: El cliente Juan Pérez ha contactado quejándose de su factura.
Su teléfono es el 600-123-456 y pide que enviemos el contrato a juan.perez88@email.com.
La agente María López está gestionando el caso.
"""

def anonimizar_datos(texto):
    # --- TU CÓDIGO AQUÍ ---
    # 1. Usa re.sub para ocultar el email. Pista regex email: r'[\w\.-]+@[\w\.-]+\.\w+'
    patron_email = r'[\w\.-]+@[\w\.-]+\.\w+'
    texto = re.sub(patron_email, '[EMAIL]', texto)
    # 2. Usa re.sub para el teléfono. Pista regex teléfono: r'\b\d{3}[-\s]?\d{3}[-\s]?\d{3}\b'
    patron_telefono = r'\b\d{3}[-\s]?\d{3}[-\s]?\d{3}\b'
    texto = re.sub(patron_telefono, '[TELÉFONO]', texto)
    # 3. Pasa el texto resultante por nlp() y busca entidades "PER".
    doc = nlp(texto)
    for ent in doc.ents:
        if ent.label_ == "PER":
    # 4. Usa texto.replace() para sustituir el texto de la entidad por '[PERSONA]'.
            texto = texto.replace(ent.text, '[PERSONA]')
    return texto

print("--- RETO 5 ---")
print(anonimizar_datos(texto_confidencial))
print("\n")

--- RETO 5 ---

Incidencia #4092: El cliente [PERSONA] ha contactado quejándose de su factura.
Su teléfono es el [TELÉFONO] y pide que enviemos el contrato a [EMAIL].
La agente [PERSONA] está gestionando el caso.





### RETO 6 - Extracción de Patrones Sintácticos (Causa-Efecto)
OBJETIVO: Extraer pares (Sustantivo, Adjetivo) utilizando el árbol de dependencias.

In [ ]:
reseñas_hardware = [
    "La batería dura poco, pero la pantalla OLED es brillante y espectacular.",
    "El teclado es incómodo para escribir textos largos.",
    "Tiene un procesador rápido, aunque el ventilador es ruidoso."
]

print("--- RETO 6 ---")
for reseña in reseñas_hardware:
    doc_hw = nlp(reseña)
    extracciones = []

    # --- TU CÓDIGO AQUÍ ---
    # 1. Itera sobre los tokens. Si el token es un Sustantivo ("NOUN"):
    for token in doc_hw:
        print("Token: ", token)
        if token.pos_ == "NOUN":
            sustantivo = token.lemma_.lower()
    # 2. CASO A: Mira a sus "hijos" (token.children). Si alguno es Adjetivo ("ADJ"), guárdalo.
            for hijo in token.children:
                print("Hijo: ", hijo)
                if hijo.pos_ == "ADJ":
                    extracciones.append((sustantivo, hijo.lemma_.lower()))
                    print("Extracciones: ", extracciones)
                    
    # 3. CASO B (Avanzado): Si el token es el sujeto de la frase (token.dep_ == "nsubj"),
    #    busca su "padre" o cabeza (token.head), que suele ser el verbo. Luego mira los hijos
    #    del verbo buscando un atributo (hijo.dep_ == "acomp" o hijo.pos_ == "ADJ").
            if token.dep_ == "nsubj":
                verbo = token.head
                print("Sujeto: ", token)
                print("Verbo: ", verbo)
                for hijo in verbo.children:
                    if hijo.dep_ == "acomp" or hijo.pos_ == "ADJ":
                        extracciones.append((sustantivo, hijo.lemma_.lower()))

    print(f"Reseña: '{reseña}'")
    print(f"  -> Atributos extraídos: {extracciones}")

--- RETO 6 ---
Token:  La
Token:  batería
Hijo:  La
Hijo:  dura
Extracciones:  [('batería', 'duro')]
Sujeto:  batería
Verbo:  poco
Token:  dura
Token:  poco
Token:  ,
Token:  pero
Token:  la
Token:  pantalla
Hijo:  la
Hijo:  OLED
Sujeto:  pantalla
Verbo:  brillante
Token:  OLED
Token:  es
Token:  brillante
Token:  y
Token:  espectacular
Token:  .
Reseña: 'La batería dura poco, pero la pantalla OLED es brillante y espectacular.'
  -> Atributos extraídos: [('batería', 'duro'), ('batería', 'brillante'), ('pantalla', 'espectacular')]
Token:  El
Token:  teclado
Hijo:  El
Sujeto:  teclado
Verbo:  incómodo
Token:  es
Token:  incómodo
Token:  para
Token:  escribir
Token:  textos
Hijo:  largos
Extracciones:  [('texto', 'largo')]
Token:  largos
Token:  .
Reseña: 'El teclado es incómodo para escribir textos largos.'
  -> Atributos extraídos: [('texto', 'largo')]
Token:  Tiene
Token:  un
Token:  procesador
Hijo:  un
Hijo:  rápido
Extracciones:  [('procesador', 'rápido')]
Token:  rápido
Token:  ,
T